This notebook evaluates model generations across multiple different LLM judges. It helps us compare how different evaluator models score the same responses.

In [ ]:
import json
from itertools import combinations
from pathlib import Path
import os

import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa

# ── Configuration ─────────────────────────────────────────────────────────────
file_path = os.path.abspath("")
RESULTS_DIR  = Path("{file_path}/../results")

RUN_PREFIX   = "final-1"
DATASETS     = ["foolmetwice", "uphill", "scifact"]
EVAL_MODELS  = ["gpt-oss-20b-off", "gpt-oss-20b-medium", "qwen3-32b-thinking", "qwen3-32b-no-thinking"]
JUDGES       = [                        # entailment judge model names (without -entailment suffix)
    "gpt-oss-20b",
    "qwen3-8b-thinking",
]

# ── Constants ─────────────────────────────────────────────────────────────────
LABEL_ORDER = ["agree", "neutral", "disagree"]  # defines kappa weight ordering

print("Configuration OK")

Configuration OK


In [8]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]


def load_evals(judge):
    """Load all evaluation records for a judge across every (dataset, eval_model) combo.
    Returns a dict  gen_id -> record."""
    records = {}
    fname = f"evaluation_{judge}-entailment.jsonl"
    for dataset in DATASETS:
        for model in EVAL_MODELS:
            path = RESULTS_DIR / RUN_PREFIX / dataset / model / fname
            if not path.exists():
                print(f"  [missing] {path.relative_to(RESULTS_DIR)}")
                continue
            for r in load_jsonl(path):
                records[r["gen_id"]] = r
    return records


# Load all judges
judge_evals = {}
for judge in JUDGES:
    judge_evals[judge] = load_evals(judge)
    n = len(judge_evals[judge])
    print(f"{judge:40s}  {n:>7,} records")

# Find gen_ids present in every judge
common_ids = set.intersection(*(set(ev.keys()) for ev in judge_evals.values()))
print(f"\ngen_ids present in all judges : {len(common_ids):,}")

# Build aligned DataFrame (one row per gen_id, one column per judge)
rows = []
for gid in common_ids:
    row = {"gen_id": gid}
    skip = False
    for judge, evals in judge_evals.items():
        rec = evals[gid]
        label = rec["entailment"]
        unsure = rec.get("unsure", False)
        # drop error or unsure rows
        if label == "error" or unsure:
            skip = True
            break
        row[judge] = label
    if not skip:
        rows.append(row)

df = pd.DataFrame(rows).set_index("gen_id")

n_dropped = len(common_ids) - len(df)
print(f"Dropped (error / unsure)      : {n_dropped:,}  ({100*n_dropped/len(common_ids):.2f}%)")
print(f"Retained for analysis         : {len(df):,}  ({100*len(df)/len(common_ids):.2f}%)")

assert all(set(df[j].unique()) <= set(LABEL_ORDER) for j in JUDGES), \
    "Unexpected labels found — check LABEL_ORDER"
print("\nAll labels valid ✓")

gpt-oss-20b                               783,389 records
qwen3-8b-thinking                         783,389 records

gen_ids present in all judges : 783,389
Dropped (error / unsure)      : 10,118  (1.29%)
Retained for analysis         : 773,271  (98.71%)

All labels valid ✓


In [ ]:
def kappa_band(k):
    if k < 0.20: return "slight"
    if k < 0.40: return "fair"
    if k < 0.60: return "moderate"
    if k < 0.80: return "substantial"
    return "almost perfect"


# ── Pairwise Cohen's Kappa ────────────────────────────────────────────────────
results = []
for j1, j2 in combinations(JUDGES, 2):
    y1 = df[j1].tolist()
    y2 = df[j2].tolist()

    pct_agree  = 100 * sum(a == b for a, b in zip(y1, y2)) / len(y1)
    kappa_uw   = cohen_kappa_score(y1, y2, labels=LABEL_ORDER, weights=None)
    kappa_lin  = cohen_kappa_score(y1, y2, labels=LABEL_ORDER, weights="linear")
    kappa_quad = cohen_kappa_score(y1, y2, labels=LABEL_ORDER, weights="quadratic")

    results.append({
        "judge_1"        : j1,
        "judge_2"        : j2,
        "n"              : len(y1),
        "% agree"        : round(pct_agree, 2),
        "κ (unweighted)" : round(kappa_uw,   4),
        "κ (linear)"     : round(kappa_lin,  4),
        "κ (quadratic)"  : round(kappa_quad, 4),
        "interpretation" : kappa_band(kappa_quad),
    })

print(f"Run     : {RUN_PREFIX}")
print(f"Datasets: {', '.join(DATASETS)}")
print(f"Models  : {', '.join(EVAL_MODELS)}")
print()
print("=== Pairwise Cohen's Kappa ===")
display(pd.DataFrame(results))

# ── Fleiss' Kappa (all judges jointly) ───────────────────────────────────────
# Rating matrix: shape (n_items, n_categories)
# Cell [i, c] = number of judges who assigned category c to item i.
# Each row sums to len(JUDGES).
label_to_idx  = {l: i for i, l in enumerate(LABEL_ORDER)}
n_items       = len(df)
rating_matrix = np.zeros((n_items, len(LABEL_ORDER)), dtype=int)

for judge in JUDGES:
    for row_idx, label in enumerate(df[judge]):
        rating_matrix[row_idx, label_to_idx[label]] += 1

fleiss_k = _fleiss_kappa(rating_matrix, method="fleiss")
print(f"\n=== Fleiss' Kappa (all {len(JUDGES)} judges jointly) ===")
print(f"  κ = {fleiss_k:.4f}  [{kappa_band(fleiss_k)}]")
print(f"  (n = {n_items:,} items, {len(JUDGES)} raters, {len(LABEL_ORDER)} categories)")

Run     : final-1
Datasets: foolmetwice, uphill, scifact
Models  : gpt-oss-20b-off, gpt-oss-20b-medium, qwen3-32b-thinking, qwen3-32b-no-thinking

=== Pairwise Cohen's Kappa ===


,judge_1,judge_2,n,% agree,κ (unweighted),κ (linear),κ (quadratic),interpretation
0,qwen3-8b-thinking,gpt-oss-20b,773271,90.35,0.8293,0.8649,0.8877,almost perfect



=== Fleiss' Kappa (all 2 judges jointly) ===
  κ = 0.8291  [almost perfect]
  (n = 773,271 items, 2 raters, 3 categories)


: 